In [1]:
"""
Step 2: Data Cleaning
=====================
"""

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# 1. LOAD RAW DATA
print("\n1. LOADING RAW DATA...")
print("-" * 80)

df = pd.read_csv('../data/raw/BD_growth_prog_anon.csv')
initial_rows = len(df)
print(f"✓ Loaded {initial_rows:,} records")

# Convert dates
df['date'] = pd.to_datetime(df['date'])
df['dob'] = pd.to_datetime(df['dob'])


1. LOADING RAW DATA...
--------------------------------------------------------------------------------
✓ Loaded 486,267 records


In [3]:
# 2. REMOVE RECORDS WITH CRITICAL ISSUES
print("\n2. REMOVING RECORDS WITH CRITICAL ISSUES")
print("-" * 80)

before = len(df)
df = df[df['flag_no_match'] == 0].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records without birth date (flag_no_match)")

before = len(df)
df = df[df['flag_under_zero'] == 0].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with negative age (flag_under_zero)")

before = len(df)
df = df[(df['height'] > 0) & (df['weight'] > 0)].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with invalid height/weight (<=0)")

before = len(df)
df = df[
    (df['height'] >= 40) & (df['height'] <= 200) &
    (df['weight'] >= 1) & (df['weight'] <= 100)     
].copy()
removed = before - len(df)
print(f"✓ Removed {removed:,} records with extreme outliers")


2. REMOVING RECORDS WITH CRITICAL ISSUES
--------------------------------------------------------------------------------
✓ Removed 795 records without birth date (flag_no_match)
✓ Removed 6 records with negative age (flag_under_zero)
✓ Removed 3,124 records with invalid height/weight (<=0)
✓ Removed 2,015 records with extreme outliers


In [4]:
# 3. HANDLE DUPLICATE MEASUREMENTS
print("\n3. HANDLING DUPLICATE MEASUREMENTS")
print("-" * 80)

df['age_days'] = (df['date'] - df['dob']).dt.days
df['age_months'] = df['age_days'] / 30.44

before = len(df)
df = df.sort_values(['child_id', 'date', 'flag_obs_number'])
df = df.drop_duplicates(subset=['child_id', 'date'], keep='first')
removed = before - len(df)
print(f"✓ Removed {removed:,} duplicate measurements on same day")

before = len(df)
df['year_quarter'] = df['date'].dt.to_period('Q')
df = df.sort_values(['child_id', 'year_quarter', 'flag_obs_number'])
df = df.drop_duplicates(subset=['child_id', 'year_quarter'], keep='first')
removed = before - len(df)
print(f"✓ Removed {removed:,} duplicate measurements in same quarter")


3. HANDLING DUPLICATE MEASUREMENTS
--------------------------------------------------------------------------------
✓ Removed 160,901 duplicate measurements on same day
✓ Removed 0 duplicate measurements in same quarter


In [5]:
# 4. HANDLE CHILDREN WITH MULTIPLE DOBs
print("\n4. HANDLING CHILDREN WITH INCONSISTENT BIRTH DATES")
print("-" * 80)

children_diff_dob = df[df['flag_different_dob'] == 1]['child_id'].unique()
if len(children_diff_dob) > 0:
    for child_id in children_diff_dob:
        child_data = df[df['child_id'] == child_id]
        most_common_dob = child_data['dob'].mode()[0]
        df = df[~((df['child_id'] == child_id) & (df['dob'] != most_common_dob))]
    print(f"✓ Standardized birth dates for {len(children_diff_dob):,} children")


4. HANDLING CHILDREN WITH INCONSISTENT BIRTH DATES
--------------------------------------------------------------------------------
✓ Standardized birth dates for 55 children


In [6]:
# 5. BIOLOGICAL CONSISTENCY CHECK (CRITICAL UPDATE)
print("\n5. REMOVING BIOLOGICAL IMPOSSIBILITIES (HEIGHT DROPS)")
print("-" * 80)

df = df.sort_values(['child_id', 'age_months'])
df['height_diff'] = df.groupby('child_id')['height'].diff()
HEIGHT_LOSS_THRESHOLD = -0.5 
invalid_children = df[df['height_diff'] < HEIGHT_LOSS_THRESHOLD]['child_id'].unique()

before = len(df)
df = df[~df['child_id'].isin(invalid_children)].copy()
removed = before - len(df)

print(f"✓ Identified {len(invalid_children):,} children with impossible height drops (shrinking)")
print(f"✓ Removed {removed:,} records to ensure monotonic growth history")
df = df.drop(columns=['height_diff'])


5. REMOVING BIOLOGICAL IMPOSSIBILITIES (HEIGHT DROPS)
--------------------------------------------------------------------------------
✓ Identified 31,116 children with impossible height drops (shrinking)
✓ Removed 192,901 records to ensure monotonic growth history


In [7]:
# 6. HANDLE WHO FLAG OUTLIERS
print("\n6. HANDLING WHO FLAG OUTLIERS")
print("-" * 80)

z_cols = ['zlen', 'zwei', 'zwfl', 'zbmi']
before = len(df)
for col in z_cols:
    if col in df.columns:
        df = df[(df[col] >= -5) & (df[col] <= 5)]
removed = before - len(df)

df['has_who_flag'] = ((df['flen'] == 1) | (df['fwei'] == 1) | 
                       (df['fwfl'] == 1) | (df['fbmi'] == 1)).astype(int)
print(f"✓ Removed {removed:,} records with extreme Z-scores (>5 or <-5)")
print("✓ Marked remaining WHO flags for model awareness")


6. HANDLING WHO FLAG OUTLIERS
--------------------------------------------------------------------------------
✓ Removed 14,425 records with extreme Z-scores (>5 or <-5)
✓ Marked remaining WHO flags for model awareness


In [8]:
# 7. FILTER BY MINIMUM MEASUREMENTS
print("\n7. FILTERING CHILDREN BY MEASUREMENT COUNT")
print("-" * 80)

MIN_MEASUREMENTS = 2
measurements_per_child = df.groupby('child_id').size()
valid_children = measurements_per_child[measurements_per_child >= MIN_MEASUREMENTS].index

before = len(df)
df = df[df['child_id'].isin(valid_children)].copy()
removed = before - len(df)

print(f"✓ Removed {removed:,} records from children with < {MIN_MEASUREMENTS} measurements")
print(f"✓ Retained {df['child_id'].nunique():,} children with sufficient data")


7. FILTERING CHILDREN BY MEASUREMENT COUNT
--------------------------------------------------------------------------------
✓ Removed 7,492 records from children with < 2 measurements
✓ Retained 24,174 children with sufficient data


In [9]:
# 8. FINALIZING DATASET
print("\n8. FINALIZING DATASET")
print("-" * 80)

df = df.sort_values(['child_id', 'date']).reset_index(drop=True)
df['measurement_number'] = df.groupby('child_id').cumcount() + 1

essential_vars = [
    'child_id', 'hh_id', 'gender', 'dob',
    'district', 'upazila', 'union', 'village',
    'date', 'height', 'weight', 'cbmi',
    'zlen', 'zwei', 'zwfl', 'zbmi',
    'has_who_flag', 'age_days', 'age_months', 'measurement_number'
]

df_clean = df[essential_vars].copy()
print(f"Final clean dataset: {len(df_clean):,} records, {df_clean['child_id'].nunique():,} children")



8. FINALIZING DATASET
--------------------------------------------------------------------------------
Final clean dataset: 104,608 records, 24,174 children


In [10]:
# 9. SAVE CLEANED DATA
print("\n9. SAVING CLEANED DATA")
print("-" * 80)

df_clean.to_csv('../data/processed/cleaned_data.csv', index=False)
print("✓ Saved to: ../data/processed/cleaned_data.csv")

# Save summary statistics
summary_stats = {
    'initial_records': initial_rows,
    'final_records': len(df_clean),
    'records_removed': initial_rows - len(df_clean),
    'unique_children': df_clean['child_id'].nunique(),
    'avg_measurements_per_child': len(df_clean) / df_clean['child_id'].nunique(),
    'cleaning_date': str(datetime.now())
}
pd.Series(summary_stats).to_csv('../data/processed/cleaning_summary.txt')



9. SAVING CLEANED DATA
--------------------------------------------------------------------------------
✓ Saved to: ../data/processed/cleaned_data.csv
